[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/11_multiagente/11_proyecto_multiagente.ipynb)

# Proyecto 11 · SupportOps multi-agente: trazar un reembolso de extremo a extremo y contener una cascada

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | Ninguno obligatorio (todo se evalúa en memoria). Opcional: `--profile core` para mirar tus trazas en Grafana (paso final) |
| **RAM aproximada** | < 600 MB (sin stack) |
| **Prerrequisitos** | Lección `11_multiagente` (propagación W3C, orquestador, cascadas) |
| **Conexión** | Reutiliza `supportops.multiagent` (sobre el `OTelInstrumentor` del módulo 03) y deja listo el sistema que el módulo 12 (MCP) y el 13 (costes) siguen ampliando |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. El equipo migró a multi-agente hace una semana y el on-call reporta tres cosas:

1. **Reembolsos que tardan más de 40 s** y acaban escalados a una persona, justo cuando el proveedor de pagos tuvo una caída.
2. **Subagentes que aparecen como trazas sueltas** en Tempo desde que `orders-agent` empezó a consumir de una cola.
3. **El coste por ticket se ha multiplicado** y nadie sabe si es el modelo grande, los reintentos o el contexto.

Tu trabajo: **(a)** propagar el contexto por la cola, **(b)** detectar trazas huérfanas, **(c)** atribuir el coste sin contar dos veces, **(d)** encontrar la causa raíz y el **trabajo zombi** del incidente, **(e)** cuantificar el desperdicio y **(f)** elegir una política de delegación que contenga la cascada.

## Datos

Todo es sintético y determinista (semilla 42, modo `mock`, tiempo simulado): el **Paso 0** te entrega tres lotes ya ejecutados con `MultiAgentSupportOps` y la telemetría de **todos** los servicios en memoria (`mesh.span_records()`, esquema `SPAN_SCHEMA` + `links`):

| Lote | Qué es |
|---|---|
| `ROWS_OK`, `RES_OK` | 20 reembolsos sin fallos, propagación correcta |
| `ROWS_NONE` | los mismos 20 reembolsos con la propagación **rota** (`propagation="none"`) |
| `ROWS_B`, `RES_B`, `WL` | el **incidente**: 30 reembolsos, ventana de fallos en `issue_refund`, política de **producción** (timeout 12 s, 2 reintentos) |

Cadena de datos: stack local (opcional) → memoria. No hace falta Docker para obtener la nota.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución; `mock`, semilla 42) |
|---|---|---|---|
| R1 | `publish` / `consume` | En 10 mensajes y los modos `parent` y `link`: trazas, padres, enlaces, tipo de span y **espera en cola** como la spec | 10/10 en ambos modos |
| R2 | `find_orphans`, `fragmentation` | En `ROWS_NONE`: nº de raíces huérfanas y trazas por petición; en `ROWS_OK`: 0 y 1,0 | coincidencia exacta |
| R3 | `leaf_totals`, `naive_totals` | Tokens por servicio = los de `RES_OK` (0 de error) y el total ingenuo ≈ 2× (doble contabilidad) | exacto / ratio 2,00 ± 0,02 |
| R4 | `root_cause`, `zombie_seconds` | La causa raíz y el síntoma de **todas** las trazas con timeout; segundos zombi = los medidos por el sistema | 100 % / ± 0,05 s |
| R5 | `amplification` | Intentos a facturación por ticket afectado (esperado: los de la política de producción) y fracción del gasto desperdiciada en intentos abandonados | exacto / ± 0,01 |
| R6 | `MY_POLICY` | Con **tu** política, sobre el mismo incidente: p95 de afectados ≤ 20 s, coste ≤ 7,5 milésimas USD/ticket (la de producción gasta ≈ 10,5), 0 degradados/error, 0 zombi, ≤ 5 tickets sanos escalados | todas a la vez |

> 📏 Los umbrales salen de la solución de referencia (y de las políticas B/C/D de la lección), medida con las semillas 42, 7 y 99: no dependen de tu máquina (el tiempo es simulado).

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
if str(ROOT) not in sys.path:   # un editable recién instalado no es visible hasta reiniciar el kernel: lo añadimos a mano
    sys.path.insert(0, str(ROOT))
import importlib; importlib.invalidate_caches()
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import dataclasses, json, re, threading, contextvars
from concurrent.futures import ThreadPoolExecutor
from opentelemetry import context as otel_context, trace as otel_trace
from opentelemetry.trace import SpanKind
from supportops.multiagent import (ORCH, SPECIALISTS, DelegationPolicy, MultiAgentSupportOps, RemoteAgentProcess, agent_tree_text,
                                   build_mesh, cost_rollup, critical_path, extract_manual, extract_sdk, format_traceparent,
                                   inject_manual, inject_sdk, parse_traceparent, service_palette, span_frame, waterfall_services)
STACK = check_stack("core")
LIVE = not STACK.degraded                      # True: el Collector responde y podemos enviar telemetría de verdad

from opentelemetry.context import Context
from opentelemetry.trace import Link
from supportops.multiagent import span_rows
from supportops.telemetry import setup_telemetry

## Paso 0 · Los tres lotes (te lo damos hecho)

El incidente: durante una **ventana contigua** el proveedor de pagos hace que `issue_refund` tarde 9 s en vencer (fallo **persistente** en `billing-agent`). Ejecutamos el mismo lote con la política de producción.

In [ ]:
sa = SupportAgent()
pool = generate_tickets(120, seed=SEED, fault_rates=None)
rf_ok = [t for t in pool if t.intent_true == "refund" and t.order_id and "issue_refund" in sa.run(t).tools_used]
def lote(n, t0=SIM_EPOCH_S, window=None):
    bad = [FaultSpec("tool_timeout", {"tool": "issue_refund", "timeout_s": 9.0, "agent": "billing-agent", "sticky": True})]
    return [dataclasses.replace(t, ticket_id=f"{t.ticket_id}-p{i}", created_at=t0 + i * 20, faults=bad if (window and i in window) else [])
            for i, t in enumerate((rf_ok * n)[:n])]

def ejecuta(tickets, policy=None, propagation="sdk"):
    mesh = build_mesh(mode="memory")
    res = MultiAgentSupportOps(mesh, policy=policy, propagation=propagation).run_many(tickets)
    return res, mesh.span_records()

OK_TK = lote(20)
RES_OK, ROWS_OK = ejecuta(OK_TK)
_, ROWS_NONE = ejecuta(OK_TK, propagation="none")
WL = lote(30, window=range(7, 22))
AFFECTED = np.array([bool(t.faults) for t in WL])
POLICY_B = DelegationPolicy(timeout_s=12, max_retries=2)
RES_B, ROWS_B = ejecuta(WL, POLICY_B)
print(f"OK: {len(RES_OK)} tickets · {len(ROWS_OK)} spans | NONE: {len(ROWS_NONE)} spans | INCIDENTE: {len(RES_B)} tickets ({AFFECTED.sum()} afectados) · {len(ROWS_B)} spans")
dfB = pd.DataFrame([r.to_row() for r in RES_B]); display(dfB[AFFECTED].head(3)[["ticket_id", "status", "latency_s", "attempts", "timeouts", "cost_usd", "zombie_s"]].round(3))

## Paso 1 · Propagar el contexto por una cola (productor y consumidor)

`orders-agent` consume tareas de una cola. Escribe las dos mitades. Reglas (spec de mensajería de OpenTelemetry, estudiadas en la sección 8 de la lección):

- **`publish(tracer, queue, payload, t_s)`**: abre un span **PRODUCER** de nombre `publish {queue}` (atributos `messaging.system="supportops-queue"`, `messaging.destination.name=queue`, `messaging.operation.type="send"`) que **empieza** en `t_s` y **termina** en `t_s + 0.01`; **inyecta** `traceparent` (`inject_manual`) con ese span activo, y devuelve el mensaje `{"payload", "headers", "queue", "published_at_s": t_s + 0.01, "message_id"}`.
- **`consume(tracer, msg, mode, start_s, duration_s)`**: abre un span **CONSUMER** `process {queue}` (atributos `messaging.system`, `messaging.destination.name`, `messaging.operation.type="process"`, `messaging.message.id`) entre `start_s` y `start_s + duration_s`. En `mode="parent"` es **hijo** del productor; en `mode="link"` abre una **traza nueva** (raíz) con un **span link** al productor. Si `mode` no es válido, lanza `ValueError`.

<details>
<summary>🪜 Pista 1</summary>

El contexto remoto se obtiene con `extract_manual(msg['headers'])`. Para «traza nueva» usa `context=Context()` (contexto vacío) al crear el span; para el link necesitas el `SpanContext` del productor: `otel_trace.get_current_span(contexto_extraído).get_span_context()`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`tracer.start_span(nombre, context=..., kind=SpanKind.CONSUMER, links=[Link(span_context)], start_time=int(t*1e9), attributes={...})`; cierra con `span.end(end_time=int(t_fin*1e9))`. Para inyectar con el productor activo: `with otel_trace.use_span(span, end_on_exit=False): headers = inject_manual()`.

</details>

In [ ]:
def publish(tracer, queue: str, payload: dict, t_s: float) -> dict:
    # TODO 1a
    raise NotImplementedError("TODO 1a")

def consume(tracer, msg: dict, mode: str, start_s: float, duration_s: float):
    # TODO 1b
    raise NotImplementedError("TODO 1b")

try:
    prod, cons = setup_telemetry("supportops-orchestrator", mode="memory", set_global=False), setup_telemetry("supportops-orders-agent", mode="memory", set_global=False)
    m = publish(prod.tracer(), "supportops.tasks", {"order": "ORD-00001"}, 100.0)
    consume(cons.tracer(), m, "parent", 101.5, 1.0)
    print(m["headers"]); display(pd.DataFrame(span_rows(prod) + span_rows(cons))[["name", "kind", "trace_id", "parent_span_id"]])
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · Detectar trazas huérfanas

**TODO 2.** Escribe:

- `find_orphans(rows)`: la lista de **spans raíz `invoke_agent`** (sin `parent_span_id`, `gen_ai.operation.name == "invoke_agent"`) cuyo servicio **no** es `supportops-orchestrator`.
- `fragmentation(rows, n_requests)`: **trazas por petición** (nº de `trace_id` distintos / peticiones). Ideal: 1,0.

<details>
<summary>🪜 Pista 1</summary>

El servicio está en `row['resource']['service.name']` y la operación en `row['attributes']['gen_ai.operation.name']`. Una traza = un `trace_id`.

</details>

In [ ]:
def find_orphans(rows: list[dict]) -> list[dict]:
    # TODO 2a
    raise NotImplementedError("TODO 2a")

def fragmentation(rows: list[dict], n_requests: int) -> float:
    # TODO 2b
    raise NotImplementedError("TODO 2b")

try:
    for nombre, rws in (("ROWS_OK", ROWS_OK), ("ROWS_NONE", ROWS_NONE)):
        print(f"{nombre}: huérfanos={len(find_orphans(rws))} · trazas/petición={fragmentation(rws, 20):.2f}")
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 3 · Atribuir el coste sin contar dos veces

**TODO 3.** Escribe:

- `leaf_totals(rows)`: por **servicio** (`service.name`), `{"calls", "input_tokens", "output_tokens"}` contando **solo** los spans `chat` (hojas).
- `naive_totals(rows)`: el total **ingenuo** de tokens de entrada: suma de `gen_ai.usage.input_tokens` de **todos** los spans que lo traen.

<details>
<summary>🪜 Pista 1</summary>

Filtra por `attributes['gen_ai.operation.name'] == 'chat'`. Los atributos de uso pueden faltar (un `chat` que falló no los tiene): usa `.get(..., 0) or 0`.

</details>

In [ ]:
def leaf_totals(rows: list[dict]) -> dict[str, dict[str, int]]:
    # TODO 3a
    raise NotImplementedError("TODO 3a")

def naive_totals(rows: list[dict]) -> int:
    # TODO 3b
    raise NotImplementedError("TODO 3b")

try:
    LT = leaf_totals(ROWS_OK); print({k.replace("supportops-", ""): v for k, v in LT.items()})
    print("ingenuo / hojas =", round(naive_totals(ROWS_OK) / sum(v["input_tokens"] for v in LT.values()), 2))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 4 · Causa raíz y trabajo zombi del incidente

**TODO 4.** Escribe, sobre las filas de **una** traza del incidente:

- `error_chain(rows)`: lista de tuplas `(profundidad, nombre, servicio_corto, error.type)` de los spans con `status == "ERROR"`, ordenada por profundidad (el servicio corto es el nombre sin el prefijo `supportops-`).
- `root_cause(rows)`: devuelve `(sintoma, causa)`; cada uno es `(nombre, servicio_corto, error.type)`. **Síntoma** = el error **más alto** del árbol; **causa** = el **más profundo**. Si no hay errores, `(None, None)`.

Y sobre las filas de **todo el lote**:

- `zombie_seconds(rows)`: suma de segundos que los subagentes siguieron trabajando **después** de que su llamador (span CLIENT con `error.type == "timeout"`) abandonara: para cada uno de esos CLIENT, `max(fin_del_hijo − fin_del_CLIENT, 0)` (el hijo es el span cuyo `parent_span_id` es el del CLIENT).

<details>
<summary>🪜 Pista 1</summary>

Profundidad: sube por `parent_span_id` hasta llegar a la raíz (`by_id = {r['span_id']: r for r in rows}`). Las trazas de una lista de filas se separan con `trace_id`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Estructura: `errs = sorted([r for r in rows if r['status']=='ERROR'], key=depth)`; síntoma = `errs[0]`, causa = `errs[-1]`. Para el zombi: indexa los hijos por padre y recorre los CLIENT en error `timeout`.

</details>

In [ ]:
def error_chain(rows: list[dict]) -> list[tuple]:
    # TODO 4a
    raise NotImplementedError("TODO 4a")

def root_cause(rows: list[dict]):
    # TODO 4b
    raise NotImplementedError("TODO 4b")

def zombie_seconds(rows: list[dict]) -> float:
    # TODO 4c
    raise NotImplementedError("TODO 4c")

try:
    tid = next(r["trace_id"] for r in ROWS_B if r["attributes"].get("error.type") == "timeout")
    print(root_cause([r for r in ROWS_B if r["trace_id"] == tid]))
    print(f"zombi total: {zombie_seconds(ROWS_B):.2f} s (el sistema midió {sum(r.zombie_s for r in RES_B):.2f} s)")
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 5 · Cuantificar el desperdicio

**TODO 5.** `amplification(rows, n_affected)` devuelve un `dict` con:

- `attempts_per_ticket`: nº de spans CLIENT `invoke_agent billing-agent` **de las trazas que contienen algún `timeout`**, dividido por `n_affected` (los tickets afectados por la ventana).
- `wasted_cost_share`: fracción del coste total de los `chat` que se gastó **dentro de intentos abandonados**, es decir, en el subárbol de un CLIENT con `error.type == "timeout"`. Precio: `supportops.pricing.cost_usd(modelo, tokens_in, tokens_out)` con el modelo de `gen_ai.response.model`.
- `theoretical`: `(r+1)^d` con `r = 2` reintentos y `d = 1` nivel que reintenta (los intentos que *debería* recibir facturación según la política).

<details>
<summary>🪜 Pista 1</summary>

Un `chat` pertenece a un subárbol abandonado si alguno de sus **ancestros** es un CLIENT con `error.type == 'timeout'`. Calcula el conjunto de `span_id` de esos CLIENT y sube por los padres de cada `chat`.

</details>

In [ ]:
from supportops.pricing import cost_usd

def amplification(rows: list[dict], n_affected: int) -> dict:
    # TODO 5
    raise NotImplementedError("TODO 5")

try:
    print(amplification(ROWS_B, int(AFFECTED.sum())))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 6 · Contener la cascada: tu política

**TODO 6.** Diseña `MY_POLICY = DelegationPolicy(...)` para el **mismo incidente**. Parámetros: `timeout_s`, `max_retries`, `backoff_s`, `breaker_threshold`, `breaker_cooldown_s`, `propagate_deadline`, `cancel_in_flight`, `fallback_to_human`.
Objetivos (los comprueba la rúbrica): bajar el **p95 de los tickets afectados**, bajar el **coste**, **no dejar tickets degradados ni en error**, **cero trabajo zombi** y **limitar el daño colateral** (tickets sanos escalados por error).
Piensa: ¿qué plazo es coherente con un especialista cuyo peor caso son 14 s? ¿Tiene sentido reintentar un fallo **persistente**? ¿Qué cuesta un disyuntor que tarda en cerrarse?

<details>
<summary>🪜 Pista 1</summary>

El plazo del llamador debe poder **cortar** al llamado (no esperar su timeout interno de 9 s): pon un `timeout_s` claramente por debajo de 14 s y activa `propagate_deadline` y `cancel_in_flight` para no dejar trabajo zombi.

</details>

<details>
<summary>🪜 Pista 2</summary>

Los reintentos no ayudan en un fallo persistente: `max_retries=0` o 1. Para que el ticket no se quede degradado, deja `fallback_to_human=True`. El disyuntor (`breaker_threshold` ≥ 2) convierte los fallos repetidos en *fallo rápido*; un `breaker_cooldown_s` largo escala tickets sanos.

</details>

In [ ]:
MY_POLICY = None
# TODO 6: MY_POLICY = DelegationPolicy(...)

if MY_POLICY is None:
    print("Pendiente: define MY_POLICY")
else:
    res_m, _ = ejecuta(WL, MY_POLICY)
    d = pd.DataFrame([r.to_row() for r in res_m])
    print(f"p95 afectados={np.quantile(d.latency_s[AFFECTED], .95):.1f}s · coste medio={d.cost_usd.mean() * 1000:.2f} milésimas USD · estados={d.status.value_counts().to_dict()}")

## Paso 7 · Rúbrica

In [ ]:
REF = {"orphans_none": 60, "frag_none": 4.0, "n_timeout_traces": 15, "attempts_per_ticket": 3.0, "wasted_share": 0.6719,
       "max_p95": 20.0, "max_cost": 7.5, "max_collateral": 5}     # calibrado con la solución (semillas 42, 7, 99)

def run_rubric():
    rows = []
    def add(r, d, ok, det=""): rows.append({"criterio": r, "descripción": d, "resultado": "✅ OK" if ok else "❌ FALLA", "detalle": det})
    # R1
    try:
        ok = {"parent": 0, "link": 0}
        for mode in ("parent", "link"):
            for i in range(10):
                p = setup_telemetry("supportops-orchestrator", mode="memory", set_global=False); c = setup_telemetry("supportops-orders-agent", mode="memory", set_global=False)
                t0 = 1000.0 + 10 * i
                m = publish(p.tracer(), "supportops.tasks", {"i": i}, t0)
                consume(c.tracer(), m, mode, t0 + 0.01 + 2.5, 1.0)
                ps, cs = span_rows(p)[0], span_rows(c)[0]
                good = (ps["kind"] == "PRODUCER" and ps["name"] == "publish supportops.tasks" and cs["kind"] == "CONSUMER" and cs["name"] == "process supportops.tasks"
                        and ps["attributes"].get("messaging.operation.type") == "send" and cs["attributes"].get("messaging.operation.type") == "process"
                        and abs((cs["start_unix_nano"] - ps["end_unix_nano"]) / 1e9 - 2.5) < 1e-3 and abs((cs["end_unix_nano"] - cs["start_unix_nano"]) / 1e9 - 1.0) < 1e-3
                        and "traceparent" in m["headers"] and abs(m["published_at_s"] - (t0 + 0.01)) < 1e-9)
                if mode == "parent":
                    good = good and cs["trace_id"] == ps["trace_id"] and cs["parent_span_id"] == ps["span_id"] and not cs["links"]
                else:
                    good = good and cs["trace_id"] != ps["trace_id"] and cs["parent_span_id"] is None and len(cs["links"]) == 1 and cs["links"][0]["span_id"] == ps["span_id"]
                ok[mode] += bool(good)
        try: consume(setup_telemetry("x", mode="memory", set_global=False).tracer(), m, "otro", 0, 1); bad_mode = False
        except ValueError: bad_mode = True
        add("R1", "publish/consume: parent y link correctos en 10/10 y ValueError con modo inválido", ok == {"parent": 10, "link": 10} and bad_mode, f"{ok} · ValueError={bad_mode}")
    except Exception as e: add("R1", "publish/consume", False, f"{type(e).__name__}: {e}")
    # R2
    try:
        o_none, o_ok = find_orphans(ROWS_NONE), find_orphans(ROWS_OK)
        f_none, f_ok = fragmentation(ROWS_NONE, 20), fragmentation(ROWS_OK, 20)
        ok = len(o_none) == REF["orphans_none"] and len(o_ok) == 0 and abs(f_none - REF["frag_none"]) < 1e-6 and abs(f_ok - 1.0) < 1e-6
        add("R2", "huérfanos y trazas por petición (NONE vs OK)", ok, f"huérfanos NONE={len(o_none)} (ref {REF['orphans_none']}) · trazas/pet NONE={f_none:.2f} (ref {REF['frag_none']:.2f}) · OK={len(o_ok)}/{f_ok:.2f}")
    except Exception as e: add("R2", "huérfanos", False, f"{type(e).__name__}: {e}")
    # R3
    try:
        lt = leaf_totals(ROWS_OK)
        tin, tout = sum(v["input_tokens"] for v in lt.values()), sum(v["output_tokens"] for v in lt.values())
        ratio = naive_totals(ROWS_OK) / tin
        ok = tin == sum(r.input_tokens for r in RES_OK) and tout == sum(r.output_tokens for r in RES_OK) and abs(ratio - 2.0) < 0.02 and sum(v["calls"] for v in lt.values()) == sum(r.n_llm_calls for r in RES_OK)
        add("R3", "totales por hojas = totales del sistema; ingenuo ≈ 2×", ok, f"hojas={tin}+{tout} (sistema {sum(r.input_tokens for r in RES_OK)}+{sum(r.output_tokens for r in RES_OK)}) · ratio={ratio:.2f}")
    except Exception as e: add("R3", "totales", False, f"{type(e).__name__}: {e}")
    # R4
    try:
        tids = sorted({r["trace_id"] for r in ROWS_B if r["attributes"].get("error.type") == "timeout"})
        good = 0
        for t in tids:
            sym, cause = root_cause([r for r in ROWS_B if r["trace_id"] == t])
            good += sym == ("invoke_agent billing-agent", "orchestrator", "timeout") and cause == ("execute_tool issue_refund", "billing-agent", "timeout")
        z, zr = zombie_seconds(ROWS_B), sum(r.zombie_s for r in RES_B)
        add("R4", "causa raíz y síntoma en el 100 % de las trazas con timeout; zombi ± 0,05 s", len(tids) == REF["n_timeout_traces"] and good == len(tids) and abs(z - zr) <= 0.05 and zr > 0, f"{good}/{len(tids)} trazas · zombi={z:.2f}s vs {zr:.2f}s")
    except Exception as e: add("R4", "causa raíz y zombi", False, f"{type(e).__name__}: {e}")
    # R5
    try:
        a = amplification(ROWS_B, int(AFFECTED.sum()))
        ok = abs(a["attempts_per_ticket"] - REF["attempts_per_ticket"]) < 1e-9 and abs(a["wasted_cost_share"] - REF["wasted_share"]) <= 0.01 and a["theoretical"] == 3
        add("R5", "amplificación: intentos por ticket, desperdicio y (r+1)^d", ok, f"{a} · ref intentos={REF['attempts_per_ticket']} desperdicio≈{REF['wasted_share']:.3f}")
    except Exception as e: add("R5", "amplificación", False, f"{type(e).__name__}: {e}")
    # R6
    try:
        assert MY_POLICY is not None, "define MY_POLICY"
        res_m, _ = ejecuta(WL, MY_POLICY); d = pd.DataFrame([r.to_row() for r in res_m])
        m = {"p95": np.quantile(d.latency_s[AFFECTED], .95), "coste": d.cost_usd.mean() * 1000, "malos": int(d.status.isin(["degraded", "error"]).sum()),
             "zombi": d.zombie_s.sum(), "colateral": int(((d.status == "escalated") & ~AFFECTED).sum())}
        checks = {f"p95 afectados ≤ {REF['max_p95']} s": m["p95"] <= REF["max_p95"], f"coste ≤ {REF['max_cost']:.2f} milésimas": m["coste"] <= REF["max_cost"],
                  "0 degradados/error": m["malos"] == 0, "zombi = 0": m["zombi"] == 0, f"colateral ≤ {REF['max_collateral']}": m["colateral"] <= REF["max_collateral"]}
        add("R6", "tu política contiene la cascada", all(checks.values()), f"{ {k: bool(v) for k, v in checks.items()} } · medido: p95={m['p95']:.1f}s coste={m['coste']:.2f} malos={m['malos']} zombi={m['zombi']:.1f} colateral={m['colateral']}")
    except Exception as e: add("R6", "política", False, f"{type(e).__name__}: {e}")
    return pd.DataFrame(rows).set_index("criterio")

In [ ]:
# Ejecuta esta celda cuando hayas terminado los TODO
display(run_rubric())

---
## Paso final (opcional) · Míralo en Grafana

Con el stack arriba (`docker compose --profile core --profile metrics up -d`), repite el lote del incidente con `build_mesh(mode="otlp")` y `generate_tickets(..., start_s="live")` (los tiempos de `lote()` usan el origen simulado de 2026: Tempo rechazaría esos datos tan antiguos; cambia `t0=` a `time.time() - 600`). Abre el dashboard **«11 · Multi-agente»** y comprueba:

1. El panel de **huérfanos** está a cero con `propagation="sdk"` y se llena con `"none"`.
2. `supportops_outcome="timeout"` solo en `billing-agent`.
3. Pulsa un `trace_id` de la tabla de errores y localiza **a ojo** el síntoma y la causa que tu `root_cause` devolvió.

---
## Retos extra (nivel experto)

1. **Handoff:** repite el incidente con `mode="handoff"`. ¿Cambia la causa raíz? ¿Qué span usarías ahora como «latencia de la petición» y por qué?
2. **Presupuesto de reintentos:** implementa un limitador (cubo de fichas) que permita reintentar como mucho el 10 % de las delegaciones en los últimos 60 s simulados y compáralo con el disyuntor.
3. **Consulta TraceQL de la cascada:** escribe (y prueba en Tempo) la consulta que devuelve las trazas con ≥ 3 CLIENT `invoke_agent billing-agent` (pista: `count()` sobre un selector con `kind = client`).
4. **Servicio de grafo:** calcula el *service graph* del incidente (llamadas y errores por arista) y compáralo con el de `ROWS_OK`.
5. **Detección temprana:** diseña una alerta sobre `supportops.delegation.attempts` por ticket (¿qué umbral y ventana detectan el incidente en < 2 min con ≤ 1 falso positivo en el lote sano?).

## Reflexión (producción e hilo conductor)

- En tu política, ¿qué parte del ahorro viene de **no reintentar** y cuál de **cancelar en vuelo**? ¿Qué dato de la traza te lo demuestra?
- ¿Cómo cambia el diseño si `billing-agent` **no** soporta cancelación (la operación de pago es no idempotente)? ¿Qué riesgos de duplicar un reembolso aparecen al reintentar y cómo los vigilarías con telemetría?
- El módulo 12 sustituye las herramientas locales por **servidores MCP remotos**: ¿qué fronteras nuevas de propagación y de timeout se añaden al árbol que acabas de analizar?
- El módulo 13 atribuirá coste por cliente, feature y modelo: ¿qué atributos faltan hoy en los spans para hacerlo sin cardinalidad explosiva?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable (redefine las funciones y vuelve a lanzar la rúbrica).

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
def publish(tracer, queue, payload, t_s):
    span = tracer.start_span(f"publish {queue}", kind=SpanKind.PRODUCER, start_time=int(t_s * 1e9), attributes={
        "messaging.system": "supportops-queue", "messaging.destination.name": queue, "messaging.operation.type": "send", "messaging.operation.name": "publish"})
    with otel_trace.use_span(span, end_on_exit=False):
        headers = inject_manual()
    span.end(end_time=int((t_s + 0.01) * 1e9))
    return {"payload": payload, "headers": headers, "queue": queue, "published_at_s": t_s + 0.01,
            "message_id": "msg-" + format(span.get_span_context().span_id, "016x")[:8]}

def consume(tracer, msg, mode, start_s, duration_s):
    if mode not in ("parent", "link"):
        raise ValueError("mode debe ser 'parent' o 'link'")
    remote = extract_manual(msg["headers"])
    if mode == "parent":
        ctx, links = remote, []
    else:
        ctx, links = Context(), [Link(otel_trace.get_current_span(remote).get_span_context())]
    span = tracer.start_span(f"process {msg['queue']}", context=ctx, kind=SpanKind.CONSUMER, links=links, start_time=int(start_s * 1e9), attributes={
        "messaging.system": "supportops-queue", "messaging.destination.name": msg["queue"], "messaging.operation.type": "process",
        "messaging.message.id": msg["message_id"]})
    span.end(end_time=int((start_s + duration_s) * 1e9))
    return span

def _svc(r): return r["resource"].get("service.name", "?")
def _op(r): return r["attributes"].get("gen_ai.operation.name")

def find_orphans(rows):
    return [r for r in rows if r["parent_span_id"] is None and _op(r) == "invoke_agent" and _svc(r) != "supportops-orchestrator"]

def fragmentation(rows, n_requests):
    return len({r["trace_id"] for r in rows}) / n_requests

def leaf_totals(rows):
    out = {}
    for r in rows:
        if _op(r) != "chat": continue
        d = out.setdefault(_svc(r), {"calls": 0, "input_tokens": 0, "output_tokens": 0})
        d["calls"] += 1
        d["input_tokens"] += r["attributes"].get("gen_ai.usage.input_tokens", 0) or 0
        d["output_tokens"] += r["attributes"].get("gen_ai.usage.output_tokens", 0) or 0
    return out

def naive_totals(rows):
    return sum(r["attributes"].get("gen_ai.usage.input_tokens", 0) or 0 for r in rows)

def error_chain(rows):
    by_id = {r["span_id"]: r for r in rows}
    def depth(r):
        d = 0
        while r["parent_span_id"] in by_id:
            r, d = by_id[r["parent_span_id"]], d + 1
        return d
    errs = sorted([r for r in rows if r["status"] == "ERROR"], key=depth)
    return [(depth(r), r["name"], _svc(r).replace("supportops-", ""), r["attributes"].get("error.type")) for r in errs]

def root_cause(rows):
    ch = error_chain(rows)
    if not ch: return None, None
    return ch[0][1:], ch[-1][1:]

def zombie_seconds(rows):
    kids = {}
    for r in rows: kids.setdefault(r["parent_span_id"], []).append(r)
    total = 0.0
    for r in rows:
        if r["kind"] == "CLIENT" and r["attributes"].get("error.type") == "timeout":
            for k in kids.get(r["span_id"], []):
                total += max(k["end_unix_nano"] - r["end_unix_nano"], 0) / 1e9
    return total

def amplification(rows, n_affected):
    by_id = {r["span_id"]: r for r in rows}
    dead = {r["span_id"] for r in rows if r["kind"] == "CLIENT" and r["attributes"].get("error.type") == "timeout"}
    def abandoned(r):
        while r["parent_span_id"] in by_id:
            r = by_id[r["parent_span_id"]]
            if r["span_id"] in dead: return True
        return False
    total = wasted = 0.0
    for r in rows:
        if _op(r) != "chat": continue
        a = r["attributes"]
        c = cost_usd(a.get("gen_ai.response.model") or a.get("gen_ai.request.model", ""), a.get("gen_ai.usage.input_tokens", 0) or 0, a.get("gen_ai.usage.output_tokens", 0) or 0)
        total += c; wasted += c if abandoned(r) else 0.0
    bad_traces = {r["trace_id"] for r in rows if r["attributes"].get("error.type") == "timeout"}
    n_bill = sum(1 for r in rows if r["kind"] == "CLIENT" and r["name"] == "invoke_agent billing-agent" and r["trace_id"] in bad_traces)
    return {"attempts_per_ticket": n_bill / n_affected, "wasted_cost_share": wasted / total, "theoretical": (2 + 1) ** 1}

MY_POLICY = DelegationPolicy(timeout_s=6, max_retries=0, propagate_deadline=True, cancel_in_flight=True, breaker_threshold=2, breaker_cooldown_s=40)
display(run_rubric())